# Seleccion de Variables para Modelo Final de Fraude

Objetivo:
- Mantener **todas las nuevas variables engineered** (A*, T*, L*).
- Evaluar en profundidad las variables del dataset original respecto a `is_fraud`.
- Recomendar que variables originales descartar para entrenamiento final.

Criterios de decision:
1. Riesgo de fuga de informacion (leakage) o sobreajuste por identificadores.
2. Calidad de dato (nulos, cardinalidad extrema, varianza nula).
3. Poder predictivo univariado (Mutual Information + Information Value).

In [1]:
import warnings
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

RANDOM_STATE = 42
TARGET_COL = "is_fraud"

FE_PATH = Path("credit_card_transactions_fe.csv")
RAW_PATH = Path("credit_card_transactions.csv")

print(f"Dataset FE existe: {FE_PATH.exists()} -> {FE_PATH.resolve()}")
print(f"Dataset RAW existe: {RAW_PATH.exists()} -> {RAW_PATH.resolve()}")

Dataset FE existe: True -> C:\Users\nicog\Documents\Proyectos Personales\FraudShield\credit_card_transactions_fe.csv
Dataset RAW existe: True -> C:\Users\nicog\Documents\Proyectos Personales\FraudShield\credit_card_transactions.csv


In [2]:
# Carga de dataset enriquecido
if not FE_PATH.exists():
    raise FileNotFoundError("No se encontro credit_card_transactions_fe.csv. Ejecuta primero el notebook de feature engineering.")

df_fe = pd.read_csv(FE_PATH)

if TARGET_COL not in df_fe.columns:
    raise ValueError(f"La columna objetivo {TARGET_COL!r} no esta en el dataset enriquecido.")

# Columnas originales tomadas desde el dataset base
if RAW_PATH.exists():
    raw_cols = pd.read_csv(RAW_PATH, nrows=0).columns.tolist()
    original_cols = [c for c in raw_cols if c in df_fe.columns and c != TARGET_COL]
else:
    # Fallback: todo lo que no sea engineered ni target
    original_cols = [
        c for c in df_fe.columns
        if c != TARGET_COL and not re.match(r"^[ATL]\d+_", c)
    ]

engineered_cols = sorted([c for c in df_fe.columns if re.match(r"^[ATL]\d+_", c)])

print(f"Filas FE: {len(df_fe):,}")
print(f"Columnas FE: {df_fe.shape[1]:,}")
print(f"Variables originales detectadas: {len(original_cols)}")
print(f"Variables engineered obligatorias detectadas: {len(engineered_cols)}")

display(pd.DataFrame({"original_cols": original_cols}).head(30))
display(pd.DataFrame({"engineered_cols": engineered_cols}).head(40))

Filas FE: 1,296,675
Columnas FE: 64
Variables originales detectadas: 23
Variables engineered obligatorias detectadas: 37


,original_cols
0,Unnamed: 0
1,trans_date_trans_time
2,cc_num
3,merchant
4,category
5,amt
6,first
7,last
8,gender
9,street


,engineered_cols
0,A10_user_merchant_cum_std
1,A11_user_merchant_cum_sum
2,A12_user_merchant_global_mean
3,A13_user_merchant_global_std
4,A14_user_merchant_global_sum
5,A15_amt_within_2std_user_merchant
6,A16_amt_within_confidence_user_merchant
7,A1_amt_diff_prev
8,A2_amt_user_cum_mean
9,A3_amt_dev_user_cum_median


In [3]:
# 1) Calidad de dato de variables originales
quality_rows = []
for col in original_cols:
    s = df_fe[col]
    quality_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "null_pct": s.isna().mean() * 100,
        "n_unique": s.nunique(dropna=True),
        "unique_ratio": s.nunique(dropna=True) / max(len(s), 1),
        "mode_pct": s.value_counts(normalize=True, dropna=False).iloc[0] * 100 if len(s) else np.nan,
    })

quality_df = pd.DataFrame(quality_rows).sort_values(["null_pct", "unique_ratio"], ascending=False)

print("Resumen de calidad (variables originales):")
display(quality_df)

print("Top posibles problemas de cardinalidad extrema:")
display(quality_df.sort_values("unique_ratio", ascending=False).head(10))

Resumen de calidad (variables originales):


,feature,dtype,null_pct,n_unique,unique_ratio,mode_pct
22,merch_zipcode,float64,15.113502,28336,0.021853,15.113502
0,Unnamed: 0,int64,0.000000,1296675,1.000000,0.000077
18,trans_num,str,0.000000,1296675,1.000000,0.000077
21,merch_long,float64,0.000000,1275745,0.983859,0.000308
19,unix_time,int64,0.000000,1274823,0.983148,0.000308
1,trans_date_trans_time,str,0.000000,1274791,0.983123,0.000308
20,merch_lat,float64,0.000000,1247805,0.962311,0.000308
5,amt,float64,0.000000,52928,0.040818,0.041799
2,cc_num,int64,0.000000,983,0.000758,0.240847
9,street,str,0.000000,983,0.000758,0.240847


Top posibles problemas de cardinalidad extrema:


,feature,dtype,null_pct,n_unique,unique_ratio,mode_pct
0,Unnamed: 0,int64,0.000000,1296675,1.000000,0.000077
18,trans_num,str,0.000000,1296675,1.000000,0.000077
21,merch_long,float64,0.000000,1275745,0.983859,0.000308
19,unix_time,int64,0.000000,1274823,0.983148,0.000308
1,trans_date_trans_time,str,0.000000,1274791,0.983123,0.000308
20,merch_lat,float64,0.000000,1247805,0.962311,0.000308
5,amt,float64,0.000000,52928,0.040818,0.041799
22,merch_zipcode,float64,15.113502,28336,0.021853,15.113502
2,cc_num,int64,0.000000,983,0.000758,0.240847
9,street,str,0.000000,983,0.000758,0.240847


In [4]:
# 2) Poder predictivo univariado de variables originales
# Muestreo estratificado para acelerar calculo y mantener representatividad
n_max = 250_000
if len(df_fe) > n_max:
    fraud_df = df_fe[df_fe[TARGET_COL] == 1]
    nonfraud_df = df_fe[df_fe[TARGET_COL] == 0]

    n_fraud_target = min(len(fraud_df), int(n_max * fraud_df.shape[0] / len(df_fe)))
    n_nonfraud_target = min(len(nonfraud_df), n_max - n_fraud_target)

    sample_df = pd.concat([
        fraud_df.sample(n=n_fraud_target, random_state=RANDOM_STATE),
        nonfraud_df.sample(n=n_nonfraud_target, random_state=RANDOM_STATE),
    ], axis=0).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    sample_df = df_fe.copy()

print(f"Muestra para seleccion univariada: {len(sample_df):,} filas")


def compute_mi(series: pd.Series, y: pd.Series) -> float:
    """Mutual Information univariada feature -> target."""
    s = series.copy()
    try:
        if pd.api.types.is_numeric_dtype(s):
            s = pd.to_numeric(s, errors="coerce")
            s = s.fillna(s.median())
            x = s.to_numpy().reshape(-1, 1)
            return float(mutual_info_classif(x, y, discrete_features=False, random_state=RANDOM_STATE)[0])
        s = s.astype(str).fillna("MISSING")
        x, _ = pd.factorize(s)
        x = x.reshape(-1, 1)
        return float(mutual_info_classif(x, y, discrete_features=True, random_state=RANDOM_STATE)[0])
    except Exception:
        return np.nan


def compute_iv(series: pd.Series, y: pd.Series, n_bins: int = 10) -> float:
    """Information Value univariado (evento: fraude=1)."""
    df_tmp = pd.DataFrame({"x": series, "y": y}).dropna(subset=["y"]).copy()

    if df_tmp["x"].nunique(dropna=True) <= 1:
        return 0.0

    if pd.api.types.is_numeric_dtype(df_tmp["x"]) and df_tmp["x"].nunique(dropna=True) > n_bins:
        ranked = df_tmp["x"].rank(method="first")
        df_tmp["bucket"] = pd.qcut(ranked, q=n_bins, duplicates="drop")
    else:
        s = df_tmp["x"].astype(str).fillna("MISSING")
        top = s.value_counts().head(25).index
        df_tmp["bucket"] = np.where(s.isin(top), s, "OTHER")

    g = df_tmp.groupby("bucket", dropna=False)["y"].agg(total="count", bad="sum")
    g["good"] = g["total"] - g["bad"]

    # Suavizado para evitar division por cero
    g["dist_good"] = (g["good"] + 0.5) / (g["good"].sum() + 0.5 * len(g))
    g["dist_bad"] = (g["bad"] + 0.5) / (g["bad"].sum() + 0.5 * len(g))

    iv = ((g["dist_good"] - g["dist_bad"]) * np.log(g["dist_good"] / g["dist_bad"])).sum()
    return float(iv)


metrics_rows = []
y_sample = sample_df[TARGET_COL].astype(int)

for col in original_cols:
    s = sample_df[col]
    metrics_rows.append({
        "feature": col,
        "mi": compute_mi(s, y_sample),
        "iv": compute_iv(s, y_sample),
    })

metrics_df = pd.DataFrame(metrics_rows)

# Escala orientativa de IV
# <0.02: inutil | 0.02-0.1: debil | 0.1-0.3: medio | 0.3-0.5: fuerte | >0.5: sospecha de leakage

def iv_band(v):
    if pd.isna(v):
        return "na"
    if v < 0.02:
        return "inutil"
    if v < 0.10:
        return "debil"
    if v < 0.30:
        return "medio"
    if v < 0.50:
        return "fuerte"
    return "muy_fuerte_o_leakage"

metrics_df["iv_band"] = metrics_df["iv"].map(iv_band)

print("Ranking por Mutual Information:")
display(metrics_df.sort_values("mi", ascending=False))

print("Ranking por Information Value:")
display(metrics_df.sort_values("iv", ascending=False))

Muestra para seleccion univariada: 250,000 filas
Ranking por Mutual Information:


,feature,mi,iv,iv_band
18,trans_num,0.035591,0.034039,debil
1,trans_date_trans_time,0.035535,0.033473,debil
5,amt,0.017325,3.666947,muy_fuerte_o_leakage
9,street,0.007123,0.053254,debil
17,dob,0.006958,0.047299,debil
10,city,0.006028,0.036528,debil
0,Unnamed: 0,0.004569,0.075859,debil
19,unix_time,0.004530,0.075859,debil
3,merchant,0.003559,0.032099,debil
13,lat,0.003347,0.020275,debil


Ranking por Information Value:


,feature,mi,iv,iv_band
5,amt,0.017325,3.666947,muy_fuerte_o_leakage
4,category,0.002177,0.763104,muy_fuerte_o_leakage
0,Unnamed: 0,0.004569,0.075859,debil
19,unix_time,0.004530,0.075859,debil
7,last,0.002710,0.057732,debil
9,street,0.007123,0.053254,debil
17,dob,0.006958,0.047299,debil
16,job,0.003143,0.044622,debil
10,city,0.006028,0.036528,debil
18,trans_num,0.035591,0.034039,debil


In [5]:
# 3) Reglas de seleccion: mantener engineered y depurar originales
selection_df = quality_df.merge(metrics_df, on="feature", how="left")

hard_drop_by_domain = {
    # IDs / trazabilidad / PII no deseables para modelo final
    "Unnamed: 0",
    "trans_num",
    "first",
    "last",
    "street",
    "cc_num",  # identificador de cliente, util para agregaciones pero no para entrenamiento directo
    "unix_time",  # timestamp crudo (alto riesgo de sobreajuste temporal)
    "trans_date_trans_time",  # fecha cruda ya representada en features temporales
    "dob",  # usar edad derivada, no fecha de nacimiento cruda
}

hard_drop = sorted([c for c in original_cols if c in hard_drop_by_domain])

# Reglas automaticas de baja senal
low_signal = set(
    selection_df[
        (selection_df["mi"].fillna(0) < 0.0015)
        & (selection_df["iv"].fillna(0) < 0.02)
    ]["feature"].tolist()
)

high_card_low_signal = set(
    selection_df[
        (selection_df["unique_ratio"] > 0.20)
        & (selection_df["mi"].fillna(0) < 0.003)
        & (selection_df["iv"].fillna(0) < 0.05)
    ]["feature"].tolist()
)

constant_cols = set(selection_df[selection_df["n_unique"] <= 1]["feature"].tolist())

recommended_drop_original = sorted(set(hard_drop) | constant_cols | high_card_low_signal)
recommended_keep_original = sorted([c for c in original_cols if c not in recommended_drop_original])

selection_df["decision"] = np.where(
    selection_df["feature"].isin(recommended_drop_original),
    "DROP",
    "KEEP"
)

print("Variables originales recomendadas para DESCARTE:")
print(recommended_drop_original)
print("\nVariables originales recomendadas para CONSERVAR:")
print(recommended_keep_original)

print("\nTabla final de seleccion (originales):")
display(selection_df.sort_values(["decision", "mi", "iv"], ascending=[True, False, False]))

# Resumen ejecutivo
print("\nResumen ejecutivo:")
print(f"- Originales detectadas: {len(original_cols)}")
print(f"- Originales recomendadas KEEP: {len(recommended_keep_original)}")
print(f"- Originales recomendadas DROP: {len(recommended_drop_original)}")
print(f"- Engineered obligatorias KEEP: {len(engineered_cols)}")

Variables originales recomendadas para DESCARTE:
['Unnamed: 0', 'cc_num', 'dob', 'first', 'last', 'merch_lat', 'merch_long', 'street', 'trans_date_trans_time', 'trans_num', 'unix_time']

Variables originales recomendadas para CONSERVAR:
['amt', 'category', 'city', 'city_pop', 'gender', 'job', 'lat', 'long', 'merch_zipcode', 'merchant', 'state', 'zip']

Tabla final de seleccion (originales):


,feature,dtype,null_pct,n_unique,unique_ratio,mode_pct,mi,iv,iv_band,decision
2,trans_num,str,0.000000,1296675,1.000000,0.000077,0.035591,0.034039,debil,DROP
5,trans_date_trans_time,str,0.000000,1274791,0.983123,0.000308,0.035535,0.033473,debil,DROP
9,street,str,0.000000,983,0.000758,0.240847,0.007123,0.053254,debil,DROP
13,dob,str,0.000000,968,0.000747,0.434650,0.006958,0.047299,debil,DROP
1,Unnamed: 0,int64,0.000000,1296675,1.000000,0.000077,0.004569,0.075859,debil,DROP
4,unix_time,int64,0.000000,1274823,0.983148,0.000308,0.004530,0.075859,debil,DROP
18,last,str,0.000000,481,0.000371,2.220603,0.002710,0.057732,debil,DROP
8,cc_num,int64,0.000000,983,0.000758,0.240847,0.002613,0.018120,inutil,DROP
19,first,str,0.000000,352,0.000271,2.056722,0.002479,0.025885,debil,DROP
3,merch_long,float64,0.000000,1275745,0.983859,0.000308,0.000261,0.012683,inutil,DROP



Resumen ejecutivo:
- Originales detectadas: 23
- Originales recomendadas KEEP: 12
- Originales recomendadas DROP: 11
- Engineered obligatorias KEEP: 37


In [6]:
# 4) Construccion de dataset final de entrenamiento
# Regla de negocio: TODAS las engineered (A/T/L) se mantienen.
final_train_cols = recommended_keep_original + engineered_cols + [TARGET_COL]
final_train_cols = [c for c in final_train_cols if c in df_fe.columns]

# Orden estable: target al final
final_train_cols = [c for c in final_train_cols if c != TARGET_COL] + [TARGET_COL]

df_model = df_fe[final_train_cols].copy()

MODEL_PATH = Path("credit_card_transactions_model_input.csv")
REPORT_PATH = Path("variable_selection_report.csv")

# Guardar entregables
df_model.to_csv(MODEL_PATH, index=False)
selection_df.sort_values(["decision", "mi", "iv"], ascending=[True, False, False]).to_csv(REPORT_PATH, index=False)

print(f"Dataset final de modelado guardado en: {MODEL_PATH.resolve()}")
print(f"Reporte de seleccion guardado en: {REPORT_PATH.resolve()}")
print(f"Shape dataset final: {df_model.shape}")

print("\nPrimeras columnas del dataset final:")
print(df_model.columns[:25].tolist())

Dataset final de modelado guardado en: C:\Users\nicog\Documents\Proyectos Personales\FraudShield\credit_card_transactions_model_input.csv
Reporte de seleccion guardado en: C:\Users\nicog\Documents\Proyectos Personales\FraudShield\variable_selection_report.csv
Shape dataset final: (1296675, 50)

Primeras columnas del dataset final:
['amt', 'category', 'city', 'city_pop', 'gender', 'job', 'lat', 'long', 'merch_zipcode', 'merchant', 'state', 'zip', 'A10_user_merchant_cum_std', 'A11_user_merchant_cum_sum', 'A12_user_merchant_global_mean', 'A13_user_merchant_global_std', 'A14_user_merchant_global_sum', 'A15_amt_within_2std_user_merchant', 'A16_amt_within_confidence_user_merchant', 'A1_amt_diff_prev', 'A2_amt_user_cum_mean', 'A3_amt_dev_user_cum_median', 'A4_identical_amt_cnt_24h', 'A5_amt_user_std_score', 'A6_amt_within_1std_user']


## Interpretacion de salida

Lee primero la tabla `selection_df` y valida especialmente:
1. Variables en `DROP` por leakage/identificadores (`trans_num`, `cc_num`, etc.).
2. Variables con `iv` extremadamente alto (>0.5), que pueden estar capturando leakage.
3. Variables categoricas de alta cardinalidad con senal baja (costo alto, beneficio bajo).

Entregables:
- `credit_card_transactions_model_input.csv`: dataset final para entrenamiento.
- `variable_selection_report.csv`: trazabilidad de la decision por variable original.